In [ ]:
from pyspark.sql import SparkSession

# 1. Create or obtain a Spark session
spark = SparkSession.builder \
    .appName("Data_Exploration") \
    .master("local[*]") \
    .getOrCreate()

# 2. Example input path (Set the full path if the file is elsewhere)
file_path = "plates_all_anonymized.csv"

# 3. Read the CSV file
# Use inferSchema for automatic type inference
# The utf8sig filename indicates an encoding convention; Spark ordinarily reads this input as UTF-8
df_sample = spark.read.csv(file_path, header=True, inferSchema=True, encoding="utf-8")

# 4. Display the schema (Column types)
print("--- Data Schema ---")
df_sample.printSchema()

# 5. Display five rows without truncating column text
print("--- First 5 Rows ---")
df_sample.show(5, truncate=False)

# 6. Display the total record count in this sample
row_count = df_sample.count()
print(f"--- Total Rows in Sample: {row_count} ---")

In [ ]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
import os
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable

spark = SparkSession.builder \
    .appName("Optimized_Index_Builder") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "100") \
    .config("spark.shuffle.compress", "true") \
    .config("spark.shuffle.spill.compress", "true") \
    .config("spark.local.dir", "./spark_temp") \
    .getOrCreate()


INPUT_FILE = "plates_all_anonymized.csv"
INDEXED_OUTPUT_DIR = "indexed_data.parquet"
SAMPLE_PLATE_TO_TEST = 151506153

df_raw = spark.read.csv(INPUT_FILE, header=True, inferSchema=True, encoding="utf-8")

columns_to_keep = [
    "CompanyCode", "DeviceID", "Lane", "MasterPlateValue",
    "PassedTime", "ReceivedTime", "Speed", "VehicleType",
    "road_code", "road_name", "state_name", "state_code",
    "PlateValue", "event_ts"
]
df_pruned = df_raw.select(columns_to_keep)


df_clean = df_pruned.dropna(subset=["PlateValue", "event_ts", "DeviceID"])

print("--- Measuring execution BEFORE applying Index (Raw Data) ---")
start_time_unoptimized = time.time()
result_unoptimized = df_clean.filter(
    (col("PlateValue") == SAMPLE_PLATE_TO_TEST) & 
    (col("event_ts").isNotNull())
).count()
end_time_unoptimized = time.time()
time_unoptimized = end_time_unoptimized - start_time_unoptimized
print(f"Time taken without index: {time_unoptimized:.4f} seconds (Found: {result_unoptimized} records)")

print("\n--- Creating Optimized Structure (B-Tree equivalent) ---")

df_optimized = df_clean.repartition(100, "PlateValue").sortWithinPartitions("PlateValue", "event_ts")

df_optimized.write.mode("overwrite").parquet(INDEXED_OUTPUT_DIR)
print(f"Optimized data saved to: {INDEXED_OUTPUT_DIR}")

print("\n--- Measuring execution AFTER applying Index ---")

df_indexed = spark.read.parquet(INDEXED_OUTPUT_DIR)

start_time_optimized = time.time()
result_optimized = df_indexed.filter(
    (col("PlateValue") == SAMPLE_PLATE_TO_TEST) & 
    (col("event_ts").isNotNull())
).count()
end_time_optimized = time.time()
time_optimized = end_time_optimized - start_time_optimized
print(f"Time taken WITH index: {time_optimized:.4f} seconds (Found: {result_optimized} records)")

print("\n--- Summary ---")
print(f"Speedup Factor: {time_unoptimized / time_optimized:.2f}x")

In [ ]:
from pyspark.sql.functions import col, window, concat_ws, collect_set, size


df_indexed = spark.read.parquet("indexed_data.parquet")

df_items = df_indexed.withColumn("Item", concat_ws("_", col("CompanyCode"), col("DeviceID")))

def create_baskets_and_evaluate(df, delta_t):
    print(f"\n--- Processing Baskets with \u0394t = {delta_t} ---")
    
    
    baskets_raw = df.groupBy(
        col("PlateValue"),
        window(col("event_ts"), delta_t).alias("TimeWindow")
    ).agg(
       
        collect_set("Item").alias("items")
    )
    
    
    baskets_valid = baskets_raw.filter(size(col("items")) >= 3)
    
    
    total_raw = baskets_raw.count()
    total_valid = baskets_valid.count()
    
    print(f"Total raw baskets created: {total_raw}")
    print(f"Total VALID baskets (>= 3 items): {total_valid}")
    print(f"Retention rate: {(total_valid/total_raw)*100:.2f}%")
    
    return baskets_valid


baskets_30m = create_baskets_and_evaluate(df_items, "30 minutes")
baskets_1h = create_baskets_and_evaluate(df_items, "1 hour")

final_baskets = baskets_1h
final_baskets.cache()

print("\n--- Sample of Final Baskets (1 Hour Window) ---")
final_baskets.select("PlateValue", "TimeWindow.start", "TimeWindow.end", "items").show(5, truncate=False)

In [ ]:
from pyspark.ml.fpm import FPGrowth
from pyspark.sql.functions import col, size

print("--- Step 1: Filtering Noise (Data Cleaning) ---")

clean_baskets = final_baskets.filter(
    (col("PlateValue") > 1000) & 
    (size(col("items")) <= 15)
)
clean_baskets.cache()

total_clean = clean_baskets.count()
print(f"Total clean baskets ready for mining: {total_clean}")

print(f"\n--- Step 2: Running FPGrowth Algorithm ---")

MIN_SUPPORT = 0.001 
MIN_CONFIDENCE = 0.1

fpGrowth = FPGrowth(itemsCol="items", minSupport=MIN_SUPPORT, minConfidence=MIN_CONFIDENCE)

model = fpGrowth.fit(clean_baskets)

print("\n--- Step 3: Extracting 3-Itemsets ---")
frequent_itemsets = model.freqItemsets

itemsets_size_3 = frequent_itemsets.filter(size(col("items")) == 3).sort(col("freq").desc())

print(f"Top 10 Frequent 3-Itemsets (minSupport={MIN_SUPPORT}):")
itemsets_size_3.show(10, truncate=False)

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import ArrayType, StringType

print("--- Step 1: Broadcasting Top Frequent Patterns ---")
top_patterns = [row['items'] for row in itemsets_size_3.limit(50).collect()]
top_patterns_b = spark.sparkContext.broadcast(top_patterns)

print("--- Step 2: Defining UDF to Calculate Misses ---")

def find_misses(basket_items):
    basket_set = set(basket_items)
    results = {} 
    
    for pattern in top_patterns_b.value:
        pattern_set = set(pattern)
        intersect = pattern_set.intersection(basket_set)
        
        
        if len(intersect) == 3:
            for item in pattern_set:
                if item not in results: results[item] = [0, 0]
                results[item][1] += 1
                
        
        elif len(intersect) == 2:
            missed_item = (pattern_set - intersect).pop()
            if missed_item not in results: results[missed_item] = [0, 0]
            results[missed_item][0] += 1
            results[missed_item][1] += 1
            
            for item in intersect:
                if item not in results: results[item] = [0, 0]
                results[item][1] += 1
                
    
    return [f"{k}|{v[0]}|{v[1]}" for k, v in results.items()]

miss_udf = F.udf(find_misses, ArrayType(StringType()))

print("--- Step 3: Applying Logic and Aggregating Results ---")

df_eval = clean_baskets.withColumn("eval_results", miss_udf(F.col("items")))

df_exploded = df_eval.select(F.explode("eval_results").alias("eval_str"))
df_split = df_exploded.select(
    F.split("eval_str", "\|").getItem(0).alias("Item"),
    F.split("eval_str", "\|").getItem(1).cast("int").alias("MissCount"),
    F.split("eval_str", "\|").getItem(2).cast("int").alias("ExpectedCount")
)


camera_miss_rates = df_split.groupBy("Item").agg(
    F.sum("MissCount").alias("TotalMiss"),
    F.sum("ExpectedCount").alias("TotalExpected")
).withColumn("MissRate", F.round((F.col("TotalMiss") / F.col("TotalExpected")) * 100, 2))


camera_final = camera_miss_rates.withColumn("CompanyCode", F.split("Item", "_").getItem(0)) \
                                .withColumn("DeviceID", F.split("Item", "_").getItem(1))

# Aggregate results by company (Company)
company_miss_rates = camera_final.groupBy("CompanyCode").agg(
    F.sum("TotalMiss").alias("CompanyTotalMiss"),
    F.sum("TotalExpected").alias("CompanyTotalExpected")
).withColumn("CompanyMissRate", F.round((F.col("CompanyTotalMiss") / F.col("CompanyTotalExpected")) * 100, 2))

print("\n--- Top 10 Cameras with Highest Miss Rate (Minimum 100 expected passes) ---")

camera_final.filter(F.col("TotalExpected") > 100).orderBy(F.col("MissRate").desc()).show(10, truncate=False)

print("\n--- Company Performance (Ranked by Miss Rate) ---")
company_miss_rates.filter(F.col("CompanyTotalExpected") > 500).orderBy(F.col("CompanyMissRate").desc()).show(10, truncate=False)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


data = {
    'CompanyCode': ['181', '123', '117', '173'],
    'CompanyName': [
        'FardIran (Setadi)',          # 181
        'Basir Pardazesh (Setadi)',   # 123
        'Rahkam (Ostani)',            # 117
        'Basir Pardazesh (Ostani)'    # 173
    ],
    'MissRate': [24.45, 23.87, 23.59, 20.77],
    'TotalExpected': [9212, 74604, 1509875, 13238]
}

df_plot = pd.DataFrame(data)
sns.set_theme(style="whitegrid")
plt.figure(figsize=(12, 6))
ax = sns.barplot(x='MissRate', y='CompanyName', data=df_plot, palette='Reds_r')


for p in ax.patches:
    width = p.get_width()
    plt.text(width + 0.3, p.get_y() + p.get_height()/2. + 0.1, 
             f'{width:.2f}%', ha="center", fontsize=12, fontweight='bold')

plt.title('Miss Rate Comparison by Company Contractors', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Miss Rate (%) - Lower is Better', fontsize=14)
plt.ylabel('Company Name', fontsize=14)
plt.xlim(0, 30) 

plt.tight_layout()
plt.show()

In [ ]:
import pyspark.sql.functions as F
from pyspark.ml.feature import CountVectorizer, MinHashLSH

print("--- Step 1: Extracting Route Sets per Vehicle ---")

df_indexed = spark.read.parquet("indexed_data.parquet")

vehicle_routes = df_indexed.filter(
    F.col("road_code").isNotNull() & 
    (F.col("PlateValue") > 1000) 
).groupBy("PlateValue").agg(
    F.collect_set(F.col("road_code").cast("string")).alias("routes")
).filter(F.size(F.col("routes")) >= 3) 


# sampled_vehicles = vehicle_routes.sample(fraction=0.01, seed=42)

sampled_vehicles = vehicle_routes.limit(2000)
sampled_vehicles.cache()

total_vehicles = sampled_vehicles.count()
print(f"Total vehicles selected for LSH analysis: {total_vehicles}")

print("\n--- Step 2: Vectorization (Converting sets to ML Vectors) ---")

cv = CountVectorizer(inputCol="routes", outputCol="features", minDF=1.0)
cv_model = cv.fit(sampled_vehicles)
vectorized_routes = cv_model.transform(sampled_vehicles)

print("\n--- Step 3: Applying MinHash and LSH ---")

mh = MinHashLSH(inputCol="features", outputCol="hashes", numHashTables=5)
lsh_model = mh.fit(vectorized_routes)

print("\n--- Step 4: Finding Candidate Pairs and Calculating Jaccard ---")



JACCARD_DISTANCE_THRESHOLD = 0.3 

similar_pairs = lsh_model.approxSimilarityJoin(
    vectorized_routes, vectorized_routes, JACCARD_DISTANCE_THRESHOLD, distCol="JaccardDistance"
).filter(
    F.col("datasetA.PlateValue") < F.col("datasetB.PlateValue") 
)

print("\n--- Top Similar Vehicle Pairs ---")

results_df = similar_pairs.select(
    F.col("datasetA.PlateValue").alias("Vehicle_1"),
    F.col("datasetB.PlateValue").alias("Vehicle_2"),
    F.col("datasetA.routes").alias("Routes_1"),
    F.col("datasetB.routes").alias("Routes_2"),
    F.round((1 - F.col("JaccardDistance")), 3).alias("JaccardSimilarity")
).orderBy(F.col("JaccardSimilarity").desc())

results_df.show(10, truncate=False)

In [ ]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

print("--- Preparing Data for Visualization ---")


pdf_results = results_df.limit(30).toPandas()


G = nx.Graph()


for index, row in pdf_results.iterrows():
    v1 = str(row['Vehicle_1'])
    v2 = str(row['Vehicle_2'])
    sim = row['JaccardSimilarity']
    
    G.add_edge(v1, v2, weight=sim)

print("--- Drawing the LSH Similarity Network ---")

plt.figure(figsize=(12, 8))

pos = nx.spring_layout(G, k=0.6, iterations=50)

nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=1500, edgecolors='darkblue', linewidths=2)

nx.draw_networkx_edges(G, pos, width=2, alpha=0.6, edge_color='gray')

nx.draw_networkx_labels(G, pos, font_size=10, font_weight='bold', font_family='sans-serif')


plt.title("Network of Vehicles with Highly Similar Routes (LSH Candidate Pairs)", fontsize=16, fontweight='bold', pad=20)
plt.axis('off') 
plt.tight_layout()
plt.show()